In [1]:
# Import stemnent
import os
import gc

import deepmatcher as dm
import pandas as pd
import torch

In [2]:
dm.data.reset_vector_cache()
#os.environ['CUDA_VISIBLE_DEVICES']='1'

print('torch version {}'.format(torch.__version__))
print('CUDA availability {}'.format(torch.cuda.is_available()))


torch version 1.8.1+cu102
CUDA availability True


In [3]:
model = dm.MatchingModel(attr_summarizer='hybrid')
model.load_state('/home/common/aravind/deepmatch_100embedding/data/model/paper_product_staples_walmart_e10_9960_50d_state.pth')

/usr/local/lib/python3.6/dist-packages/torch/nn/modules/module.py:770: UserWarning: Using non-full backward hooks on a Module that does not take as input a single Tensor or a tuple of Tensors is deprecated and will be removed in future versions. This hook will be missing some of the grad_input. Please use register_full_backward_hook to get the documented behavior.
  warnings.warn("Using non-full backward hooks on a Module that does not take as input a "
/usr/local/lib/python3.6/dist-packages/torch/nn/modules/module.py:795: UserWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  warnings.warn("Using a non-full backward hook when the forward contains multiple autograd Nodes "


In [4]:
candidate = dm.data.process_unlabeled(
    path=os.path.join('/home/common/aravind/deepmatch_100embedding/data/populated_features', 'paper_products_features_populated.csv'),
    trained_model=model,
    ignore_columns=['left_id', 'right_id', 'label'])


Reading and processing data from "/home/common/aravind/deepmatch_100embedding/data/populated_features/paper_products_features_populated.csv"
100%|█████████▉| 399999/400000 [00:11<00:00, 35638.65it/s]


In [6]:
#result_dataframe = model.run_prediction(candidate, output_attributes=list(candidate.get_raw_table().columns))
result_dataframe = model.run_prediction(candidate, output_attributes=False)

===>  PREDICT Epoch 9


RuntimeError: input.size(-1) must be equal to input_size. Expected 100, got 50

In [ ]:
del model,candidate
gc.collect()

In [ ]:
result_dataframe.head()

In [ ]:
high_score_pairs = result_dataframe[result_dataframe['match_score'] >= 0.9].sort_values(by=['match_score'], ascending=False)
high_score_pairs.head()

In [ ]:
print(high_score_pairs.shape)

In [ ]:
descession_threshold = 0.5
result_dataframe['prediction'] = result_dataframe['match_score'].apply(lambda score: 1 if score >=descession_threshold else 0)

In [ ]:
result_dataframe['prediction'].value_counts()

In [ ]:
result_dataframe.head()

In [ ]:
result_dataframe['TP'] = result_dataframe.apply(lambda row: 1 if row['label'] ==1 and row['prediction'] == 1.0 else 0, axis=1)
result_dataframe['TN'] = result_dataframe.apply(lambda row: 1 if row['label'] ==0 and row['prediction'] == 0.0 else 0, axis=1)
result_dataframe['FP'] = result_dataframe.apply(lambda row: 1 if row['label'] ==0 and row['prediction'] == 1.0 else 0, axis=1)
result_dataframe['FN'] = result_dataframe.apply(lambda row: 1 if row['label'] ==1 and row['prediction'] == 0.0 else 0, axis=1)

In [ ]:
total_ones=result_dataframe[result_dataframe['label']==1].shape[0]
total_zeros=result_dataframe[result_dataframe['label']==0].shape[0]

print(total_ones, total_zeros)

In [ ]:
TP_COUNT= result_dataframe['TP'].sum()
TN_COUNT= result_dataframe['TN'].sum()
FP_COUNT= result_dataframe['FP'].sum()
FN_COUNT= result_dataframe['FN'].sum()
print('TP COUNT :{}'.format(TP_COUNT))
print('TN COUNT :{}'.format(TN_COUNT))
print('FP COUNT :{}'.format(FP_COUNT))
print('FN COUNT :{}'.format(FN_COUNT))

Precession = TP_COUNT/(TP_COUNT+FP_COUNT)*100
Recall = TP_COUNT/(TP_COUNT+FN_COUNT)*100
F1_score = 2*(Precession * Recall)/(Precession + Recall)
Accuracy = (TP_COUNT+TN_COUNT)/(TP_COUNT+TN_COUNT+FP_COUNT+FN_COUNT)

TPR= TP_COUNT/total_ones
TNR= TN_COUNT/total_zeros

print('Precession :{}'.format(Precession))
print('Recall :{}'.format(Recall))
print('F1_score :{}'.format(F1_score))
print('Accuracy :{}'.format(Accuracy))
print("TPR: ", TPR)
print("TNR: ", TNR)

In [ ]:
result_dataframe.drop(columns=['match_score'], inplace=True)

In [ ]:
result_dataframe.to_csv('/home/common/aravind/deepmatch_100embedding/data/Final_result/paper_product_e10_9960_100d.csv', sep='^', index=False)

In [ ]:
del result_dataframe
gc.collect()

In [ ]:
TP COUNT :430
TN COUNT :77612
FP COUNT :318
FN COUNT :369
Precession :57.48663101604278
Recall :53.81727158948686
F1_score :55.59146735617324
Accuracy :0.9912738635064589
------------------------
TP COUNT :790
TN COUNT :61583
FP COUNT :16347
FN COUNT :9
Precession :4.609908385364999
Recall :98.87359198998749
F1_score :8.809099018733273
Accuracy :0.7922493617345577    